In [0]:
create or refresh streaming table restaurant.silver.fact_reviews(
  constraint validate_review_id expect(review_id is not null) on violation fail update,
  constraint validate_sentiment expect(sentiment in ("positive","negative","neutral")) on violation drop row,
  constraint validate_rating expect(rating > 0) on violation drop row
)
as
select 
   review_id,
   order_id,
   customer_id,
   restaurant_id,
   cast(rating as int) as rating,
   review_text,
   get_json_object(analysis_json, '$.sentiment') as sentiment,
   get_json_object(analysis_json,'$.issue_delivery')::boolean as issue_delivery,
   get_json_object(analysis_json,'$.issue_delivery_reason') as issue_delivery_reason,
   get_json_object(analysis_json,'$.issue_food_quality')::boolean as issue_food_quality,
   get_json_object(analysis_json,'$.issue_food_quality_reason') as issue_food_quality_reason,
   get_json_object(analysis_json,'$.issue_pricing')::boolean as issue_pricing,
   get_json_object(analysis_json,'$.issue_pricing_reason') as issue_pricing_reason,
   get_json_object(analysis_json,'$.issue_portion_size')::boolean as issue_portion_size,
   get_json_object(analysis_json,'$.issue_portion_size_reason') as issue_portion_size_reason,
   cast(review_timestamp as timestamp) as review_timestamp,
   current_timestamp() as _ingest_timestamp 
   from (
select
    *,
    ai_query(
      'databricks-gpt-oss-20b',
      CONCAT(
        'Analyze the following review and return ONLY a valid JSON object with this exact structure: ',
        '{"sentiment": "<positive/neutral/negative>", ',
        '"issue_delivery": <true/false>, ',
        '"issue_delivery_reason": "<reason or empty string>", ',
        '"issue_food_quality": <true/false>, ',
        '"issue_food_quality_reason": "<reason or empty string>", ',
        '"issue_pricing": <true/false>, ',
        '"issue_pricing_reason": "<reason or empty string>", ',
        '"issue_portion_size": <true/false>, ',
        '"issue_portion_size_reason": "<reason or empty string>"}. ',
        'Rules: sentiment must be exactly one of: positive, neutral, negative. ',
        'Each issue field is true/false only. ',
        'Each reason field should contain a brief explanation if the issue is true, otherwise empty string. ',
        'Review text: ', review_text
      )
    ) AS analysis_json
  FROM stream(live.restaurant.bronze.customer_reviews)
   ) as ai